# MPR Market Step-by-Step Debug

This notebook runs the MPR market loop with explicit iteration logging.

For each iteration it shows:
- `q_current`
- per-job bids
- `q_clear` returned by market clearing
- `residual`


In [32]:
from pathlib import Path
from typing import Any, Dict, Optional, Tuple
import socket
import sys
import time

import numpy as np
import pandas as pd

# Ensure repo root is importable even if notebook starts outside project root.
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "mpr_int").exists() and (candidate / "job_scheduler").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import mpr_int as mpr
import job_scheduler
from job_scheduler import DEFAULT_PERF_SHEET_MAP
from job_scheduler.performance_data import load_perf_data_for_jobs

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
print("Repo root:", REPO_ROOT)


Repo root: /Users/imtiazevan/Desktop/Projects/temp/MPR_System_Beta


In [ ]:
# -----------------------------
# User-configurable variables
# -----------------------------
WORKBOOK_PATH = REPO_ROOT / "job_scheduler" / "data" / "all_model_data_by_rank.xlsx"
JOB_RANKS = {
    "xsbenchmpi": 2,
    "comd": 2,
}
TARGET_REDUCTION_W = 105

Q_BOUNDS = (0.1, 5.0)
MAX_ITERS = 100
DELTA_Q_TOL = 0.05
RESIDUAL_ABS_TOL = 5.0
ALPHA = 1.0

# Cycle detection settings (same idea as mpr_int/threaded.py)
CYCLE_MIN_PERIOD = 2
CYCLE_MAX_PERIOD = 30
CYCLE_Q_TOL = 0.05
CYCLE_PICK = "min_abs_residual"  # or "min_q"

HOST = "127.0.0.1"
PORT = 8899  # Change if busy.
USE_FREE_PORT = False
STARTUP_TIMEOUT_S = 20.0
CLIENT_START_DELAY_S = 0.05


In [34]:
def get_free_port(host: str = "127.0.0.1") -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind((host, 0))
        return int(sock.getsockname()[1])


if USE_FREE_PORT:
    PORT = get_free_port(HOST)

print("Notebook CWD:", Path.cwd())
print("Resolved workbook:", WORKBOOK_PATH)
print("Workbook exists:", WORKBOOK_PATH.exists())
job_names = list(JOB_RANKS.keys())
sheet_map = {job: DEFAULT_PERF_SHEET_MAP.get(job, job) for job in job_names}

job_perf_data, perf_audit_df = load_perf_data_for_jobs(
    xlsx_path=WORKBOOK_PATH,
    job_names=job_names,
    sheet_map=sheet_map,
)

display(perf_audit_df)
missing = [job for job in job_names if job not in job_perf_data]
if missing:
    raise RuntimeError(f"Missing perf data for jobs: {missing}")

print("Workbook:", WORKBOOK_PATH)
print("Jobs:", JOB_RANKS)
print("Target reduction (W):", TARGET_REDUCTION_W)
print(f"Server endpoint: http://{HOST}:{PORT}")


Notebook CWD: /Users/imtiazevan/Desktop/Projects/temp/MPR_System_Beta/notebooks
Resolved workbook: /Users/imtiazevan/Desktop/Projects/temp/MPR_System_Beta/job_scheduler/data/all_model_data_by_rank.xlsx
Workbook exists: True


,job,sheet,status,rows
0,xsbenchmpi,xsbench,LOADED,8
1,comd,comd,LOADED,8


Workbook: /Users/imtiazevan/Desktop/Projects/temp/MPR_System_Beta/job_scheduler/data/all_model_data_by_rank.xlsx
Jobs: {'xsbenchmpi': 2, 'comd': 2}
Target reduction (W): 60
Server endpoint: http://127.0.0.1:8899


In [39]:
def run_market_step_by_step(
    *,
    job_perf_data: Dict[str, pd.DataFrame],
    target_reduction_w: float,
    host: str,
    port: int,
    q_bounds: Tuple[float, float] = (0.1, 5.0),
    max_iters: int = 100,
    delta_q_tol: float = 0.05,
    residual_abs_tol: float = 5.0,
    alpha: float = 1.0,
    cycle_min_period: int = 2,
    cycle_max_period: int = 30,
    cycle_q_tol: float = 0.05,
    cycle_pick: str = "min_abs_residual",
    startup_timeout_s: float = 20.0,
    client_start_delay_s: float = 0.05,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    if cycle_pick not in {"min_abs_residual", "min_q"}:
        raise ValueError("cycle_pick must be 'min_abs_residual' or 'min_q'.")

    negotiator = mpr.ThreadedMPRIntNegotiator(
        job_perf_data=job_perf_data,
        target_reduction_w=float(target_reduction_w),
        q_bounds=q_bounds,
        max_iters=max_iters,
        delta_q_tol=delta_q_tol,
        residual_abs_tol=residual_abs_tol,
        alpha=alpha,
        host=host,
        port=port,
        startup_timeout_s=startup_timeout_s,
        client_start_delay_s=client_start_delay_s,
        verbose=False,
    )

    rows: list[Dict[str, float]] = []
    q_current_history: list[float] = []
    q_clear_history: list[float] = []
    residual_history: list[float] = []

    converged = False
    convergence_mode = "max_iter"
    selected_iteration: Optional[int] = None
    cycle_period: Optional[int] = None
    cycle_max_diff: Optional[float] = None

    q_current = (q_bounds[0] + q_bounds[1]) / 2.0
    t0 = time.perf_counter()

    try:
        negotiator.server.start()
        negotiator.server.wait_until_ready(timeout_s=startup_timeout_s)

        for client in negotiator.clients.values():
            client.start()
            if client_start_delay_s > 0:
                time.sleep(client_start_delay_s)

        negotiator.server.wait_for_clients(len(negotiator.clients), timeout_s=startup_timeout_s)

        missing_ack = [
            cid for cid, client in negotiator.clients.items()
            if not client.wait_until_registered(timeout_s=2.0)
        ]
        if missing_ack:
            raise TimeoutError(f"Missing registration ACK for clients: {missing_ack}")

        for iteration in range(1, max_iters + 1):
            bids = negotiator.server.request_bids(q_current, negotiator.client_ids)
            q_new, residual = negotiator.server.clearing_step(
                bids=bids,
                client_ids=negotiator.client_ids,
                target_effective_w=float(target_reduction_w),
                q_bounds=q_bounds,
            )

            delta_q = abs(q_new - q_current)
            row: Dict[str, float] = {
                "iteration": float(iteration),
                "q_current": float(q_current),
                "q_clear": float(q_new),
                "delta_q": float(delta_q),
                "residual": float(residual),
                "clearing_total_reduction_w": float(residual + target_reduction_w),
            }
            for client_id in negotiator.client_ids:
                row[f"bid_{client_id}"] = float(bids[client_id])
            rows.append(row)

            q_current_history.append(float(q_current))
            q_clear_history.append(float(q_new))
            residual_history.append(float(residual))

            # Fixed-point convergence
            if delta_q < delta_q_tol and abs(residual) < residual_abs_tol:
                converged = True
                convergence_mode = "fixed_point"
                selected_iteration = iteration
                break

            # Cycle detection on q_clear history
            detected_period: Optional[int] = None
            max_period = min(int(cycle_max_period), len(q_clear_history) // 2)
            for p in range(int(cycle_min_period), max_period + 1):
                prev = np.array(q_clear_history[-2 * p : -p], dtype=float)
                curr = np.array(q_clear_history[-p:], dtype=float)
                if prev.size != p or curr.size != p:
                    continue
                max_diff = float(np.max(np.abs(curr - prev)))
                if max_diff < float(cycle_q_tol):
                    detected_period = p
                    cycle_max_diff = max_diff
                    break

            if detected_period is not None:
                cycle_start = len(q_clear_history) - detected_period
                cycle_q = np.array(q_clear_history[cycle_start:], dtype=float)
                cycle_res = np.abs(np.array(residual_history[cycle_start:], dtype=float))
                if cycle_pick == "min_q":
                    local_idx = int(np.argmin(cycle_q))
                else:
                    local_idx = int(np.argmin(cycle_res))
                selected_iteration = cycle_start + local_idx + 1
                converged = True
                convergence_mode = "cycle"
                cycle_period = detected_period
                break

            q_current = float(np.clip((1.0 - alpha) * q_current + alpha * q_new, q_bounds[0], q_bounds[1]))

        if selected_iteration is None:
            selected_iteration = len(rows)

        iter_df = pd.DataFrame(rows)
        summary: Dict[str, Any] = {
            "converged": bool(converged),
            "convergence_mode": str(convergence_mode),
            "selected_iteration": int(selected_iteration),
            "iterations_run": int(len(rows)),
            "target_reduction_w": float(target_reduction_w),
            "cycle_period": int(cycle_period) if cycle_period is not None else None,
            "cycle_max_diff": float(cycle_max_diff) if cycle_max_diff is not None else None,
            "negotiation_time_s": float(time.perf_counter() - t0),
        }
        return iter_df, summary
    finally:
        for client in negotiator.clients.values():
            client.stop()
        for client in negotiator.clients.values():
            client.join(timeout=2.0)
        negotiator.server.shutdown()
        negotiator.server.join(timeout=2.0)


In [40]:
iter_df, summary = run_market_step_by_step(
    job_perf_data=job_perf_data,
    target_reduction_w=TARGET_REDUCTION_W,
    host=HOST,
    port=PORT,
    q_bounds=Q_BOUNDS,
    max_iters=MAX_ITERS,
    delta_q_tol=DELTA_Q_TOL,
    residual_abs_tol=RESIDUAL_ABS_TOL,
    alpha=ALPHA,
    cycle_min_period=CYCLE_MIN_PERIOD,
    cycle_max_period=CYCLE_MAX_PERIOD,
    cycle_q_tol=CYCLE_Q_TOL,
    cycle_pick=CYCLE_PICK,
    startup_timeout_s=STARTUP_TIMEOUT_S,
    client_start_delay_s=CLIENT_START_DELAY_S,
)

display(iter_df)
summary


,iteration,q_current,q_clear,delta_q,residual,clearing_total_reduction_w,bid_xsbenchmpi,bid_comd
0,1.0,2.550000,3.248189,0.698189,0.004618,120.004618,0.537541,0.012227
1,2.0,3.248189,0.159558,3.088631,0.015496,120.015496,0.006862,0.015574
2,3.0,0.159558,0.781770,0.622212,0.000236,120.000236,0.033635,0.076348
3,4.0,0.781770,3.830768,3.048998,0.003919,120.003919,0.164797,0.374070
4,5.0,3.830768,0.188448,3.642320,0.060422,120.060422,0.008092,0.018368
5,6.0,0.188448,0.923389,0.734941,0.002612,120.002612,0.039725,0.090172
6,7.0,0.923389,4.524163,3.600775,0.000110,120.000110,0.194651,0.441833
7,8.0,4.524163,0.222606,4.301557,0.067390,120.067390,0.009557,0.021692
8,9.0,0.222606,1.090690,0.868084,0.000369,120.000369,0.046926,0.106517
9,10.0,1.090690,5.000000,3.909310,-2.116169,117.883831,0.229918,0.521885


{'converged': True,
 'convergence_mode': 'cycle',
 'selected_iteration': 15,
 'iterations_run': 15,
 'target_reduction_w': 120.0,
 'cycle_period': 3,
 'cycle_max_diff': 0.0,
 'negotiation_time_s': 0.760191291999945}

In [41]:
if not iter_df.empty:
    selected_idx = int(summary["selected_iteration"]) - 1
    selected_row = iter_df.iloc[selected_idx]
    bid_cols = [c for c in iter_df.columns if c.startswith("bid_")]
    selected_bids = {c.removeprefix("bid_"): float(selected_row[c]) for c in bid_cols}
    print("Selected iteration:", summary["selected_iteration"])
    print("Selected q_current:", float(selected_row["q_current"]))
    print("Selected q_clear:", float(selected_row["q_clear"]))
    print("Selected residual:", float(selected_row["residual"]))
    print("Selected bids:", selected_bids)


Selected iteration: 15
Selected q_current: 0.24577565867903572
Selected q_clear: 1.204674876357997
Selected residual: 0.012168085758361258
Selected bids: {'xsbenchmpi': 0.05181054327779901, 'comd': 0.11760340251066828}
